In [1]:
import numpy as np
import pandas as pd

In [ ]:
startups=pd.read_csv('startup_funding.csv')

startups.rename(columns={
    'City  Location':'City',
    'Date dd/mm/yyyy':'Date',
    'Investors Name':'Investors',
    'Amount in USD':'Amount',
    'InvestmentnType':'Investment Type',
    'Startup Name':'Startup',

},inplace=True)

startups.drop(columns='Remarks',inplace=True)
startups.set_index('Sr No',inplace=True)

startups['Amount']=startups['Amount'].str.replace(r'[^0-9.]','',regex=True)
startups['Amount']=pd.to_numeric(startups['Amount'],errors='coerce')
startups['Amount']=startups['Amount'].fillna(startups['Amount'].mode()[0])


startups['Date'] = startups['Date'].str.replace('.', '/', regex=False)
startups['Date'] = startups['Date'].str.replace('//', '/')
startups['Date'] = pd.to_datetime(startups['Date'], dayfirst=True,errors='coerce')
startups['Date']=startups['Date'].fillna(pd.Timestamp('1900-01-01'))

startups['City']=startups['City'].str.replace(r'\\xc2\\xa0',"",regex=False).str.strip()

startups['City']=startups['City'].str.split('/').str[0].str.strip()

startups['City']=startups['City'].str.split(' and ').str[0].str.strip()

startups['City']=startups['City'].str.split(' & ').str[0].str.strip()

startups['City']=startups['City'].str.split(',').str[0].str.strip()



city_corrections = {
    'Gurgaon': 'Gurugram',
    'Delhi': 'New Delhi',
    'Ahemadabad': 'Ahmedabad',
    'Ahemdabad': 'Ahmedabad',
    'SFO': 'San Francisco',
    'Nw Delhi': 'New Delhi',
    'Bengaluru': 'Bangalore',
    'bangalore': 'Bangalore',
    'Kolkatta': 'Kolkata',
    'US':'USA',
    'Bhubneswar':'Bhubaneswar',
    'Kormangala': 'Bangalore',
    'Andheri':'Mumbai',
    'Chembur':'Mumbai',
    'Taramani':'Chennai',
    'Missourie':'Mussoorie',
    'Tulangan':'Telangana'
}

startups=startups.replace(city_corrections)
startups['City'].unique()


array(['Bangalore', 'Gurugram', 'New Delhi', 'Mumbai', 'Chennai', 'Pune',
       'Noida', 'Faridabad', 'San Francisco', 'San Jose', 'Amritsar',
       'Telangana', 'Hyderabad', 'Burnsville', 'Menlo Park', 'Palo Alto',
       'Santa Monica', 'Singapore', 'Nairobi', 'Haryana', 'New York',
       'Karnataka', 'Bhopal', 'India', 'Jaipur', 'Nagpur', 'Indore',
       'California', 'Ahmedabad', 'Rourkela', 'Srinagar', 'Bhubaneswar',
       'Chandigarh', 'Kolkata', 'Coimbatore', 'Udaipur', nan, 'Surat',
       'Goa', 'Uttar Pradesh', 'Gaya', 'Vadodara', 'Trivandrum',
       'Mussoorie', 'Panaji', 'Gwalior', 'Karur', 'Udupi', 'Kochi',
       'Agra', 'Hubli', 'Kerala', 'Kozhikode', 'USA', 'Siliguri',
       'Lucknow', 'Kanpur', 'London', 'Seattle', 'Varanasi', 'Jodhpur',
       'Boston', 'Belgaum', 'Dallas'], dtype=object)

Your Mission: Group the data to find out which CityLocation has received the highest total amount of funding in Indian history.

In [522]:
startups.groupby('City')['Amount'].sum().sort_values(ascending=False).head(1)

City
Bangalore    2.107391e+10
Name: Amount, dtype: float64

Your Mission: Find the top 5 investors who have funded the most number of unique startups.

In [527]:

clean_investors_df=startups.dropna(subset='Investors').copy()

clean_investors_df['Investors'] = clean_investors_df['Investors'].str.replace(r'\\xc2\\xa0|\\xe2\\x80\\x99s', '', regex=True)


clean_investors_df['Investors'] = clean_investors_df['Investors'].str.split(r',|&|\band\b', regex=True)


clean_investors_df = clean_investors_df.explode('Investors')

clean_investors_df['Investors'] = clean_investors_df['Investors'].str.strip()


clean_investors_df = clean_investors_df[clean_investors_df['Investors'] != '']

garbage_words = 'undisclosed|others|several angel investors|existing investors'
clean_investors_df = clean_investors_df[~clean_investors_df['Investors'].str.contains(garbage_words, case=False, na=False)]

clean_investors_df['Investors'].value_counts().head(5)


Investors
Accel Partners     81
Sequoia Capital    75
Blume Ventures     58
Kalaari Capital    57
SAIF Partners      51
Name: count, dtype: int64